**Stage 3 (thesis-era, unencrypted).** First Python port of the DCT steganography: hides the **whole frame** (no encryption) and applies a simplified H.264-style DCT quantisation at QP=27 before embedding. Earlier prototype kept for reference: it does **not** reproduce the thesis β-sweep numbers (measured on vid2-1: 11.9 dB reconstruction PSNR with MJPG, 16.8 dB with a lossless codec, vs. 36.5 dB in the thesis). Written for MATLAB-generated inputs (`vid3-matlab.avi`). The encrypted, bbox-only version is `Multi_Stego_and_Reconstructed_Video_Results.ipynb`.

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
import cv2
import numpy as np
from scipy.fftpack import dct, idct

print("Starting complete steganography process...")

# ==================== PARAMETERS ====================
QP = 27
Beta = 30
Alpha = 0.05

# Input files
original_video_path = '/content/vid3-matlab.avi'
isolated_persons_path = '/content/isolated_persons_vid3-matlab.avi'
bg_image_path = '/content/BG_frame.jpg'

# Output files
stego_output_path = f'/content/QP{QP}B{Beta}.avi'
reconstructed_output_path = f'/content/Rec_Msg_QP{QP}B{Beta}.avi'

# ==================== HELPER FUNCTIONS ====================

def dct2(block):
    """2D DCT"""
    return dct(dct(block.T, norm='ortho').T, norm='ortho')

def idct2(block):
    """2D inverse DCT"""
    return idct(idct(block.T, norm='ortho').T, norm='ortho')

def adjust_image_a(img, alpha):
    """
    Adjust image to prevent boundary issues during quantization
    Maps pixel values away from 0 and 1
    """
    img_double = img.astype(np.float64) / 255.0

    # Clip values away from boundaries
    img_double[img_double >= (1 - alpha)] = 1 - alpha
    img_double[img_double <= alpha] = alpha

    return img_double

def qun1(x, y, alp):
    """
    Quantization-based hiding
    x: cover (4x4 block)
    y: message (4x4 block)
    alp: quantization parameter (Beta)
    Returns: stego block
    """
    inter_val = np.linspace(0.0, 2.0, alp)
    s = np.zeros((4, 4))

    for i in range(4):
        for j in range(4):
            if x[i, j] >= 2.0:
                a = inter_val[-2]
                b = 2.0
            elif x[i, j] < 0.0:
                a = 0.0
                b = inter_val[1]
            else:
                s_index1 = np.where(inter_val <= x[i, j])[0]
                s_index2 = np.where(inter_val > x[i, j])[0]
                a = inter_val[s_index1[-1]]
                b = inter_val[s_index2[0]]

            s[i, j] = a + ((b - a) * y[i, j])

    return s

def inv_qun1(s, alp):
    """
    Inverse quantization for extraction
    s: stego block (4x4)
    alp: quantization parameter (Beta)
    Returns: extracted message block
    """
    inter_val = np.linspace(0.0, 2.0, alp)
    y = np.zeros((4, 4))

    for i in range(4):
        for j in range(4):
            if s[i, j] >= 2.0:
                a = inter_val[-2]
                b = 2.0
            elif s[i, j] < 0.0:
                a = 0.0
                b = inter_val[1]
            else:
                s_index1 = np.where(inter_val <= s[i, j])[0]
                s_index2 = np.where(inter_val > s[i, j])[0]
                a = inter_val[s_index1[-1]]
                b = inter_val[s_index2[0]]

            y[i, j] = (s[i, j] - a) / (b - a)

    return y

def h264_compression(cover, qp):
    """
    Simplified H.264-style compression
    For simplicity, we'll use a basic DCT-based quantization
    """
    mm, nn, ll = cover.shape
    com_cover = np.zeros_like(cover)

    # Process each color channel
    for p in range(ll):
        # Process in 16x16 macroblocks
        for i in range(0, mm - 15, 16):
            for j in range(0, nn - 15, 16):
                block = cover[i:i+16, j:j+16, p].astype(np.float64)

                # Apply DCT and quantization (simplified)
                # In real H.264 this is much more complex
                dct_block = dct2(block / 255.0)

                # Quantization step
                q_step = 2 ** ((qp - 12) / 6.0)
                quantized = np.round(dct_block / q_step)

                # Dequantization
                dequantized = quantized * q_step

                # Inverse DCT
                reconstructed = idct2(dequantized) * 255.0
                reconstructed = np.clip(reconstructed, 0, 255)

                com_cover[i:i+16, j:j+16, p] = reconstructed

    return com_cover.astype(np.uint8)

def quantization_hiding(comp_cover, msg, beta):
    """
    Hide message in compressed cover using quantization
    """
    mm, nn, ll = comp_cover.shape
    stego_frame = np.zeros((mm, nn, ll))

    # Process each color channel
    for p in range(ll):
        # Process in 4x4 blocks
        for i in range(0, mm - 3, 4):
            for j in range(0, nn - 3, 4):
                c_block = comp_cover[i:i+4, j:j+4, p].astype(np.float64)
                m_block = msg[i:i+4, j:j+4, p]

                # DCT of cover block
                c_inv_mb = dct2(c_block / 255.0)

                # Hide message using quantization
                s_inv_mb = qun1(np.abs(c_inv_mb), m_block, beta)

                # Preserve sign
                s_inv_mb = np.where(c_inv_mb < 0, -s_inv_mb, s_inv_mb)

                # Preserve DC coefficient
                s_inv_mb[0, 0] = c_inv_mb[0, 0]

                # Inverse DCT
                s_block = idct2(s_inv_mb) * 255.0
                s_block = np.clip(np.round(s_block), 0, 255)

                stego_frame[i:i+4, j:j+4, p] = s_block

    return stego_frame.astype(np.uint8)

def quantization_extraction(stego, beta):
    """
    Extract hidden message from stego frame
    """
    mm, nn, ll = stego.shape
    rec_msg = np.zeros((mm, nn, ll))

    # Process each color channel
    for p in range(ll):
        # Process in 4x4 blocks
        for i in range(0, mm - 3, 4):
            for j in range(0, nn - 3, 4):
                s_block = stego[i:i+4, j:j+4, p].astype(np.float64)

                # DCT of stego block
                s_mb = dct2(s_block / 255.0)

                # Extract message using inverse quantization
                rec_m_block = inv_qun1(np.abs(s_mb), beta)

                # Interpolate DC coefficient from neighbors
                rec_m_block[0, 0] = (rec_m_block[0, 1] + rec_m_block[1, 1] + rec_m_block[1, 0]) / 3.0

                rec_msg[i:i+4, j:j+4, p] = rec_m_block

    rec_msg = (rec_msg * 255).astype(np.uint8)
    return rec_msg

# ==================== EMBEDDING PROCESS ====================

print("\n=== STARTING EMBEDDING PROCESS ===")

# Load videos
original_video = cv2.VideoCapture(original_video_path)
isolated_video = cv2.VideoCapture(isolated_persons_path)
bg_image = cv2.imread(bg_image_path)

# Get video properties
frame_width = int(original_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(original_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = original_video.get(cv2.CAP_PROP_FPS)
total_frames = int(original_video.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video properties: {frame_width}x{frame_height}, FPS: {fps}, Frames: {total_frames}")

# Resize background
resized_bg = cv2.resize(bg_image, (frame_width, frame_height))

# Create video writer for stego video
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
stego_writer = cv2.VideoWriter(stego_output_path, fourcc, fps, (frame_width, frame_height))

frame_count = 0

while True:
    ret_orig, frame = original_video.read()
    ret_iso, mask_frame = isolated_video.read()

    if not ret_orig or not ret_iso:
        break

    # Store original frame as message
    msg_frame = frame.copy()

    # Resize mask if needed
    if mask_frame.shape[:2] != (frame_height, frame_width):
        mask_frame = cv2.resize(mask_frame, (frame_width, frame_height))

    # Create cover frame by replacing persons with background
    cover_frame = frame.copy().astype(np.float64)
    mask = mask_frame.astype(np.float64)

    # Replace person pixels with background (with 3x3 neighborhood)
    for i in range(frame_height - 9):
        for j in range(frame_width - 9):
            # Check if pixel is part of a person (non-black in mask)
            if not (mask[i, j, 0] < 1 and mask[i, j, 1] < 1 and mask[i, j, 2] < 1):
                # Replace 3x3 neighborhood with background
                cover_frame[i:i+3, j:j+3, :] = resized_bg[i:i+3, j:j+3, :]

    cover_frame = cover_frame.astype(np.uint8)

    # H.264 compression of cover
    comp_cover = h264_compression(cover_frame, QP)

    # Adjust message frame
    msg_adjusted = adjust_image_a(msg_frame, Alpha)

    # Hide message in compressed cover
    stego_frame = quantization_hiding(comp_cover, msg_adjusted, Beta)

    # Write stego frame
    stego_writer.write(stego_frame)

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Embedded {frame_count}/{total_frames} frames")

original_video.release()
isolated_video.release()
stego_writer.release()

print(f"\n✓ Embedding complete! Stego video saved to: {stego_output_path}")

# ==================== EXTRACTION PROCESS ====================

print("\n=== STARTING EXTRACTION PROCESS ===")

# Load stego video
stego_video = cv2.VideoCapture(stego_output_path)

# Get video properties
frame_width = int(stego_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(stego_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = stego_video.get(cv2.CAP_PROP_FPS)
total_frames = int(stego_video.get(cv2.CAP_PROP_FRAME_COUNT))

# Create video writer for reconstructed message
rec_writer = cv2.VideoWriter(reconstructed_output_path, fourcc, fps, (frame_width, frame_height))

frame_count = 0

while True:
    ret, stego_frame = stego_video.read()

    if not ret:
        break

    # Extract hidden message
    rec_msg_frame = quantization_extraction(stego_frame, Beta)

    # Message correction: preserve bright pixels from stego
    threshold_value = 200
    bright_mask = np.all(stego_frame > threshold_value, axis=2)
    rec_msg_frame[bright_mask] = stego_frame[bright_mask]

    # Write reconstructed frame
    rec_writer.write(rec_msg_frame)

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Extracted {frame_count}/{total_frames} frames")

stego_video.release()
rec_writer.release()

print(f"\n✓ Extraction complete! Reconstructed video saved to: {reconstructed_output_path}")
print("\n=== PROCESS COMPLETE ===")

In [ ]:
import cv2
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine

print("Starting results computation...")

# ==================== PARAMETERS ====================
QP = 27
Beta = 30

# Input files
original_video_path = '/content/vid3-matlab.avi'
inpainted_video_path = '/content/vid3-inpainted-matlab.avi'
stego_video_path = f'/content/QP{QP}B{Beta}.avi'
reconstructed_video_path = f'/content/Rec_Msg_QP{QP}B{Beta}.avi'

# Output Excel file
output_excel = f'/content/ResultsQP{QP}.xlsx'
sheet_name = f'QP{QP}B{Beta}'

# ==================== HELPER FUNCTIONS ====================

def matrix_to_row_vector(mat):
    """Convert matrix to row vector (column-wise flattening)"""
    return mat.flatten('F')  # 'F' means column-major (Fortran-style) order

def compute_img_mse(img_s, img_n):
    """
    Compute MSE for images S & N
    N is noisy image and S is uncorrupted image
    """
    img_s = img_s.astype(np.float64)
    img_n = img_n.astype(np.float64)

    # Compute difference for each channel
    dr = (img_s[:, :, 0] - img_n[:, :, 0]) ** 2  # Red channel
    dg = (img_s[:, :, 1] - img_n[:, :, 1]) ** 2  # Green channel
    db = (img_s[:, :, 2] - img_n[:, :, 2]) ** 2  # Blue channel

    # Total difference
    d = dr + dg + db

    # Compute MSE
    m, n = img_s.shape[:2]
    mse = np.sum(d) / (3 * m * n)

    return mse

def compute_img_psnr(img_s, img_n):
    """
    Compute PSNR for images S & N
    """
    mse = compute_img_mse(img_s, img_n)

    p = 255  # Max value in image

    if mse == 0:
        psnr = float('inf')
    else:
        psnr = 10 * np.log10(p**2 / mse)

    return mse, psnr

def img_similarity(img_a, img_b):
    """
    Compute Normalized Similarity measure between two images A & B
    A is the original image, and B is the attacked one
    Returns similarity percentage
    """
    # Convert images to row vectors
    aa = matrix_to_row_vector(img_a).astype(np.float64)
    bb = matrix_to_row_vector(img_b).astype(np.float64)

    # Compute normalized similarity
    # Sim = (AA·BB / ||BB||) / (||AA|| / ||AA||)
    # Which simplifies to: (AA·BB) / (||AA|| * ||BB||)

    dot_aa_bb = np.dot(aa, bb)
    norm_aa = np.sqrt(np.dot(aa, aa))
    norm_bb = np.sqrt(np.dot(bb, bb))

    if norm_aa == 0 or norm_bb == 0:
        return 0.0

    # Ordinary similarity
    sim = dot_aa_bb / norm_bb

    # Normalized similarity
    sim = sim / norm_aa

    # Convert to percentage
    sim = sim * 100.0

    return sim

# ==================== LOAD VIDEOS ====================

print("\nLoading videos...")
video1 = cv2.VideoCapture(original_video_path)
video2 = cv2.VideoCapture(inpainted_video_path)
video3 = cv2.VideoCapture(stego_video_path)
video4 = cv2.VideoCapture(reconstructed_video_path)

# Get frame counts
num_frames1 = int(video1.get(cv2.CAP_PROP_FRAME_COUNT))
num_frames2 = int(video2.get(cv2.CAP_PROP_FRAME_COUNT))
num_frames3 = int(video3.get(cv2.CAP_PROP_FRAME_COUNT))
num_frames4 = int(video4.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Frame counts:")
print(f"  Original (frame1): {num_frames1}")
print(f"  Inpainted (frame2): {num_frames2}")
print(f"  Stego (frame3): {num_frames3}")
print(f"  Reconstructed (frame4): {num_frames4}")

# Use minimum number of frames
num_frames = min([num_frames1, num_frames2, num_frames3, num_frames4])
print(f"\nProcessing {num_frames} frames...")

# ==================== PROCESS FRAMES ====================

# Initialize results list
results = []

# Add header row
results.append({
    'Frame': 'Frame',
    'MSE (Stego)': 'MSE (Stego)',
    'PSNR (Stego)': 'PSNR (Stego)',
    'Similarity (Stego)': 'Similarity (Stego)',
    'MSE (Extracted)': 'MSE (Extracted)',
    'PSNR (Extracted)': 'PSNR (Extracted)',
    'Similarity (Extracted)': 'Similarity (Extracted)',
    'Extra Column': 'Extra Column'
})

for kk in range(num_frames):
    # Read frames
    ret1, frame1 = video1.read()  # Original message frame
    ret2, frame2 = video2.read()  # Cover frame (inpainted)
    ret3, frame3 = video3.read()  # Stego frame
    ret4, frame4 = video4.read()  # Reconstructed message frame

    if not (ret1 and ret2 and ret3 and ret4):
        print(f"Warning: Could not read frame {kk+1}")
        break

    # Debug output
    if (kk + 1) % 10 == 0 or kk == 0:
        print(f"\nFrame {kk + 1}:")
        print(f"  frame1 size: {frame1.shape}")
        print(f"  frame2 size: {frame2.shape}")
        print(f"  frame3 size: {frame3.shape}")
        print(f"  frame4 size: {frame4.shape}")

    # Check if all frames have the same size
    if not (frame1.shape == frame2.shape == frame3.shape == frame4.shape):
        print(f"Error: Frame size mismatch at frame {kk+1}")
        print(f"  frame1: {frame1.shape}, frame2: {frame2.shape}")
        print(f"  frame3: {frame3.shape}, frame4: {frame4.shape}")

        # Resize frames to match frame1 size
        target_size = (frame1.shape[1], frame1.shape[0])  # (width, height)
        if frame2.shape != frame1.shape:
            frame2 = cv2.resize(frame2, target_size)
        if frame3.shape != frame1.shape:
            frame3 = cv2.resize(frame3, target_size)
        if frame4.shape != frame1.shape:
            frame4 = cv2.resize(frame4, target_size)
        print(f"  Frames resized to: {frame1.shape}")

    # Stego test metrics (compare inpainted cover with stego)
    mse_stego, psnr_stego = compute_img_psnr(frame2, frame3)
    sim_stego = img_similarity(frame2, frame3)

    # Extracted test metrics (compare original with reconstructed)
    mse_extracted, psnr_extracted = compute_img_psnr(frame1, frame4)
    sim_extracted = img_similarity(frame1, frame4)

    # Store results
    results.append({
        'Frame': kk + 1,
        'MSE (Stego)': mse_stego,
        'PSNR (Stego)': psnr_stego,
        'Similarity (Stego)': sim_stego,
        'MSE (Extracted)': mse_extracted,
        'PSNR (Extracted)': psnr_extracted,
        'Similarity (Extracted)': sim_extracted,
        'Extra Column': ''
    })

    # Progress update
    if (kk + 1) % 30 == 0:
        print(f"Processed {kk + 1}/{num_frames} frames")
        print(f"  Stego - MSE: {mse_stego:.2f}, PSNR: {psnr_stego:.2f} dB, Sim: {sim_stego:.2f}%")
        print(f"  Extracted - MSE: {mse_extracted:.2f}, PSNR: {psnr_extracted:.2f} dB, Sim: {sim_extracted:.2f}%")

# Release video captures
video1.release()
video2.release()
video3.release()
video4.release()

# ==================== SAVE TO EXCEL ====================

print(f"\nSaving results to Excel: {output_excel}")

# Create DataFrame
df = pd.DataFrame(results)

# Write to Excel
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

print("✓ Results saved successfully!")

# ==================== COMPUTE AVERAGES ====================

# Skip the header row for calculations
numeric_results = results[1:]

if len(numeric_results) > 0:
    avg_mse_stego = np.mean([r['MSE (Stego)'] for r in numeric_results])
    avg_psnr_stego = np.mean([r['PSNR (Stego)'] for r in numeric_results])
    avg_sim_stego = np.mean([r['Similarity (Stego)'] for r in numeric_results])

    avg_mse_extracted = np.mean([r['MSE (Extracted)'] for r in numeric_results])
    avg_psnr_extracted = np.mean([r['PSNR (Extracted)'] for r in numeric_results])
    avg_sim_extracted = np.mean([r['Similarity (Extracted)'] for r in numeric_results])

    print("\n" + "="*60)
    print("AVERAGE RESULTS:")
    print("="*60)
    print("\nStego Quality (Inpainted vs Stego):")
    print(f"  Average MSE:        {avg_mse_stego:.4f}")
    print(f"  Average PSNR:       {avg_psnr_stego:.4f} dB")
    print(f"  Average Similarity: {avg_sim_stego:.4f}%")

    print("\nExtraction Quality (Original vs Reconstructed):")
    print(f"  Average MSE:        {avg_mse_extracted:.4f}")
    print(f"  Average PSNR:       {avg_psnr_extracted:.4f} dB")
    print(f"  Average Similarity: {avg_sim_extracted:.4f}%")
    print("="*60)

print("\n✓ Process complete!")

# Display download link for Colab
from google.colab import files
print("\nDownloading Excel file...")
files.download(output_excel)